In [5]:
from google.cloud import bigquery
import os

def init_bigquery_client() -> bigquery.Client:
    """Initialize BigQuery client"""
    bq_credentials_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
    project_id = os.getenv('GCP_PROJECT_ID')
    
    if not project_id:
        raise ValueError("GCP_PROJECT_ID environment variable not set")
    
    if bq_credentials_path and os.path.exists(bq_credentials_path):
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = bq_credentials_path
        print(f"Using BigQuery credentials: {bq_credentials_path}")
    
    return bigquery.Client(project=project_id)

#pulling data from bigquery tables
nodes_query = "SELECT * FROM `etl-testing-478716.instagram_graph.nodes`"
edges_query = "SELECT * FROM `etl-testing-478716.instagram_graph.edges`"
client = init_bigquery_client()
nodes_bq = client.query(nodes_query).to_dataframe()
edges_bq = client.query(edges_query).to_dataframe()

Using BigQuery credentials: etl-testing-478716-c0b6c2c512e0.json


/opt/miniconda3/envs/heyyall/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [6]:
#deduplicating nodes based on username and full_name
nodes_dedup = nodes_bq.drop_duplicates(subset = ["username", "full_name"])

#filter to public accounts only
nodes_dedup_public = nodes_dedup[nodes_dedup['is_private'] == False].reset_index(drop=True)
#split into two df's based on is_verified
nodes_verified = nodes_dedup_public[nodes_dedup_public['is_verified'] == True].reset_index(drop=True)
nodes_unverified = nodes_dedup_public[nodes_dedup_public['is_verified'] == False].reset_index(drop=True)

print(f"Total nodes: {len(nodes_dedup)}")
print(f"Verified nodes: {len(nodes_verified)}")
print(f"Unverified nodes: {len(nodes_unverified)}")

Total nodes: 5290
Verified nodes: 542
Unverified nodes: 2810


In [ ]:
from apify_client import ApifyClient
from main import run_profile_crawl

In [3]:
import time
import pandas as pd

In [7]:
print("Running the next step...")
username_list = nodes_verified['username'].tolist()

Running the next step...


In [10]:
username_list_sample = username_list[:5]
print(f"Sample usernames: {username_list_sample}")

Sample usernames: ['nickradphotog', 'endo.satine', 'itslunaaura', 'verylonely.shop', 'davecreaney']


In [11]:

profiles = run_profile_crawl(username_list_sample)

Crawling profiles for 5 accounts...


[apify.instagram-profile-scraper runId:S9pmDCenzgFbUG4hH] -> Status: RUNNING, Message: 
[apify.instagram-profile-scraper runId:S9pmDCenzgFbUG4hH] -> 2026-03-31T16:32:04.756Z ACTOR: Pulling container image of build AIf7xvwjg39uyRUcq from registry.
[apify.instagram-profile-scraper runId:S9pmDCenzgFbUG4hH] -> 2026-03-31T16:32:04.759Z ACTOR: Creating container.
[apify.instagram-profile-scraper runId:S9pmDCenzgFbUG4hH] -> 2026-03-31T16:32:04.797Z ACTOR: Starting container.
[apify.instagram-profile-scraper runId:S9pmDCenzgFbUG4hH] -> 2026-03-31T16:32:04.798Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.instagram-profile-scraper runId:S9pmDCenzgFbUG4hH] -> 2026-03-31T16:32:05.398Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v22.22.1"}
[apify.instagram-profile-scraper runId:S9pmDCenzgFbUG4hH] -> 2026-03-31T16:32:05.567Z INFO  Results Limit [object Object], ACTOR_MAX_PAID_DATASET_ITEMS
[

Fetched profiles for 5 accounts


In [14]:

# Turning profiles into dataframe and saving as parquet file
profiles_df = pd.DataFrame(profiles)
profiles_df['Ratio'] = profiles_df['followersCount'] / profiles_df['followsCount'].replace(0, 1)  # Avoid division by zero
# run_id = str(int(time.time()))
# profiles_df.to_parquet(f"profiles_{run_id}.parquet", index=False)
# print(f"Saved profiles with ratio as parquet file locally")

print("Top 5 profiles by ratio and total followers:")
top_profiles = profiles_df.sort_values(by=["Ratio", "followersCount"], ascending=False).head(5)
print(top_profiles[["username", "followersCount", "followsCount", "Ratio"]])

Top 5 profiles by ratio and total followers:
          username  followersCount  followsCount      Ratio
4  verylonely.shop           24164           752  32.132979
2      itslunaaura           10762          1246   8.637239
1    nickradphotog            5212          1046   4.982792
0      davecreaney            4168          3057   1.363428
3      endo.satine            2448          1815   1.348760


In [ ]:
#running crawl on selected accounts 
from main import store_to_bigquery
from main import run_bfs_crawl

seed_accounts = []


run_id = str(int(time.time()))
G = run_bfs_crawl(seed_accounts)
#before storing save as parquet files and save locally as backup 
# Insert data
nodes = [{"username": n, **d} for n, d in G.nodes(data=True)]
edges = [{"source": u, "target": v, **d} for u, v, d in G.edges(data=True)]

# Save as parquet files locally
nodes_df = pd.DataFrame(nodes)
edges_df = pd.DataFrame(edges)
nodes_df.to_parquet(f"nodes_{run_id}.parquet", index=False)
edges_df.to_parquet(f"edges_{run_id}.parquet", index=False)
print(f"Saved nodes and edges as parquet files locally")

#saving to biquery dataset
store_to_bigquery(G)

In [18]:
#visualizing network 
import networkx as nx

G = nx.DiGraph()  # Directed graph for following relationships
# Add nodes and edges to G based on your data

nodes_bq = nodes_bq.drop_duplicates(subset=['username']).reset_index(drop=True)
edges_bq = edges_bq.drop_duplicates(subset=['source', 'target']).reset_index(drop=True)

for item,row in nodes_bq.iterrows():
    G.add_node(row['username'], is_verified=row['is_verified'], is_private=row['is_private'])

for item,row in edges_bq.iterrows():
    G.add_edge(row['source'], row['target'], **row.to_dict())
import pandas as pd

# Replace pd.NA and None with empty string for all node attributes
for n, d in G.nodes(data=True):
    for k, v in list(d.items()):
        if v is pd.NA or v is None:
            d[k] = ""

# Replace pd.NA and None with empty string for all edge attributes
for u, v, d in G.edges(data=True):
    for k, v2 in list(d.items()):
        if v2 is pd.NA or v2 is None:
            d[k] = ""

nx.write_gexf(G, "my_graph.gexf")